In [1]:
!pip install networkx -q


In [2]:
"""

DATASET
--------
Kaggle "American Epilepsy Society Seizure Prediction Challenge", human
patient data (Patient_1), pre-labeled preictal/interictal .mat segments, the
same verified loader used in earlier work in this portfolio.

"""

import numpy as np
from scipy.io import loadmat
import glob
import os
import re
from typing import List, Dict, Optional

np.random.seed(42)


# ==============================================================================
# Gromov's four-point delta-hyperbolicity
# ==============================================================================

def four_point_delta(dist_matrix: np.ndarray, w: int, x: int, y: int, z: int) -> float:
    """Computes delta(w,x,y,z) for one quadruple, given a full pairwise distance matrix."""
    s1 = dist_matrix[w, x] + dist_matrix[y, z]
    s2 = dist_matrix[w, y] + dist_matrix[x, z]
    s3 = dist_matrix[w, z] + dist_matrix[x, y]
    sums = sorted([s1, s2, s3], reverse=True)
    return (sums[0] - sums[1]) / 2.0


def estimate_delta_hyperbolicity(dist_matrix: np.ndarray, n_samples: int = 2000,
                                   percentile: float = 95.0, rng: Optional[np.random.RandomState] = None) -> Dict:
    """
    Estimates delta-hyperbolicity by randomly sampling quadruples (checking
    all C(n,4) quadruples is infeasible for anything but tiny n). Returns
    both the requested percentile (the headline delta-hyperbolicity
    estimate, robust to a small number of noisy points) and the full sampled
    distribution, so the caller can inspect it directly rather than trust a
    single summary number blindly.
    """
    if rng is None:
        rng = np.random.RandomState(42)

    n = dist_matrix.shape[0]
    if n < 4:
        return {"delta_estimate": float("nan"), "samples": np.array([])}

    deltas = np.empty(n_samples)
    for i in range(n_samples):
        idx = rng.choice(n, size=4, replace=False)
        deltas[i] = four_point_delta(dist_matrix, *idx)

    return {
        "delta_estimate": float(np.percentile(deltas, percentile)),
        "delta_mean": float(deltas.mean()),
        "samples": deltas,
    }


# ==============================================================================
# Correctness check: a PERFECT TREE must give delta = 0 for every quadruple
# ==============================================================================
# This is a known analytic property of tree metrics (the four-point condition
# is, by definition, the exact characterization of when a metric embeds
# isometrically into a tree), used here as a direct correctness test of the
# implementation above, not just a plausibility check.

def build_perfect_tree_distance_matrix(n_leaves: int = 16) -> np.ndarray:
    """
    Builds the shortest-path distance matrix of a perfect binary tree with
    n_leaves leaves (all edge weights = 1), which is a tree metric by
    construction, and therefore must give delta = 0 for every quadruple.
    """
    import networkx as nx
    depth = int(np.ceil(np.log2(n_leaves)))
    G = nx.balanced_tree(r=2, h=depth)
    leaves = [node for node in G.nodes() if G.degree(node) == 1][:n_leaves]
    dist_dict = dict(nx.all_pairs_shortest_path_length(G))
    n = len(leaves)
    D = np.zeros((n, n))
    for i, li in enumerate(leaves):
        for j, lj in enumerate(leaves):
            D[i, j] = dist_dict[li][lj]
    return D


def run_correctness_check():
    print("Correctness check: delta-hyperbolicity of a perfect tree metric must be 0.")
    tree_dist = build_perfect_tree_distance_matrix(n_leaves=16)
    result = estimate_delta_hyperbolicity(tree_dist, n_samples=2000, percentile=100.0)
    print(f"  Max sampled delta on a perfect tree: {result['delta_estimate']:.8f}  (expected: 0.0)")
    assert result["delta_estimate"] < 1e-9, "Implementation error: perfect tree should give delta = 0 exactly."
    print("  PASSED: implementation matches the known analytic property of tree metrics.\n")

    print("Contrast check: delta-hyperbolicity of points on a Euclidean circle (NOT tree-like).")
    n_pts = 16
    angles = np.linspace(0, 2 * np.pi, n_pts, endpoint=False)
    circle_pts = np.stack([np.cos(angles), np.sin(angles)], axis=1)
    circle_dist = np.sqrt(((circle_pts[:, None, :] - circle_pts[None, :, :]) ** 2).sum(axis=-1))
    result_circle = estimate_delta_hyperbolicity(circle_dist, n_samples=2000, percentile=95.0)
    print(f"  95th percentile delta on a circle: {result_circle['delta_estimate']:.4f}  (expected: > 0, a circle is not tree-like)")
    assert result_circle["delta_estimate"] > 0.01, "A circle should show meaningfully nonzero delta."
    print("  PASSED: non-tree-like structure correctly produces nonzero delta.\n")


# ==============================================================================
# Population activity vectors from EEG (the analogue of place-cell population
# vectors in the original paper, substituting "moment in time" for "location")
# ==============================================================================
# Zhang et al. built population vectors per spatial LOCATION (activity of all
# CA1 neurons while the animal was in a given place). EEG has no direct
# analogue of "location", so this project substitutes short, fixed-length
# sub-windows of a seizure-related segment as the unit of comparison: each
# sub-window's multi-channel activity forms one "population state" vector,
# and the delta-hyperbolicity of the SET of these states within a segment is
# what gets compared between preictal and interictal segments.

SUBWINDOW_SECONDS = 1.0     # short enough to capture a distinct "brain state" snapshot
N_STATES_PER_SEGMENT = 40   # number of population-state vectors sampled per segment


def extract_population_states(segment_data: np.ndarray, sfreq: float,
                                n_states: int = N_STATES_PER_SEGMENT) -> np.ndarray:
    """
    segment_data: [n_channels, n_samples] for one preictal or interictal
    segment. Returns [n_states, n_channels] array: one population-activity
    vector (band power per channel, a standard, simple EEG feature) per
    sampled sub-window.
    """
    n_channels, n_samples_total = segment_data.shape
    subwindow_len = int(SUBWINDOW_SECONDS * sfreq)

    max_start = n_samples_total - subwindow_len
    if max_start <= 0:
        return np.zeros((0, n_channels))

    starts = np.linspace(0, max_start, n_states).astype(int)
    states = np.zeros((len(starts), n_channels))

    for i, start in enumerate(starts):
        chunk = segment_data[:, start:start + subwindow_len]
        # Band power (mean squared amplitude) per channel: a simple, standard
        # EEG feature, analogous in spirit to a firing-rate population vector.
        states[i] = np.mean(chunk ** 2, axis=1)

    # Log-transform and z-score per channel: EEG power is heavily right-skewed
    # and channels vary widely in baseline scale; this keeps the distance
    # computation from being dominated by a few high-amplitude channels.
    states = np.log1p(states)
    states = (states - states.mean(axis=0, keepdims=True)) / (states.std(axis=0, keepdims=True) + 1e-8)
    return states


def population_state_distance_matrix(states: np.ndarray) -> np.ndarray:
    """Euclidean distance between population-state vectors, the same
    representational-distance concept the original paper uses for place-cell
    population vectors."""
    diff = states[:, None, :] - states[None, :, :]
    return np.sqrt((diff ** 2).sum(axis=-1))


def segment_delta_hyperbolicity(segment_data: np.ndarray, sfreq: float) -> Optional[float]:
    states = extract_population_states(segment_data, sfreq)
    if states.shape[0] < 4:
        return None
    dist_matrix = population_state_distance_matrix(states)
    n_quadruples = min(2000, states.shape[0] ** 4)  # cap sampling effort sensibly for small n
    result = estimate_delta_hyperbolicity(dist_matrix, n_samples=n_quadruples, percentile=95.0)
    return result["delta_estimate"]


# ==============================================================================
# Real data loading: Kaggle American Epilepsy Society Seizure Prediction
# Challenge, human patient data (reused, verified loader from earlier work)
# ==============================================================================

DATA_ROOT = "/kaggle/input/datasets/mohitnxn/patient-1/Patient_1"
SUBJECT = "Patient_1"


def _find_data_key(mat_dict: dict) -> str:
    candidate_keys = [k for k in mat_dict.keys() if not k.startswith("__")]
    if not candidate_keys:
        raise ValueError("No non-metadata key found in .mat file.")
    return candidate_keys[0]


def load_mat_segment(file_path: str) -> Optional[Dict]:
    try:
        mat = loadmat(file_path, squeeze_me=True, struct_as_record=False)
        key = _find_data_key(mat)
        struct = mat[key]
        data = np.asarray(struct.data, dtype=np.float32)
        sfreq = float(struct.sampling_frequency)
        sequence = int(struct.sequence) if hasattr(struct, "sequence") else None
        return {"data": data, "sampling_frequency": sfreq, "sequence": sequence}
    except Exception as e:
        print(f"Could not load {file_path}: {e}")
        return None


def discover_segment_files(data_root: str, subject: str) -> Dict[str, List[str]]:
    preictal_files = sorted(glob.glob(os.path.join(data_root, f"{subject}_preictal_segment_*.mat")))
    interictal_files = sorted(glob.glob(os.path.join(data_root, f"{subject}_interictal_segment_*.mat")))
    print(f"Found {len(preictal_files)} preictal and {len(interictal_files)} interictal files for {subject}.")
    return {"preictal": preictal_files, "interictal": interictal_files}


# ==============================================================================
# Seizure-block grouping for preictal files (the leakage/independence fix)
# ==============================================================================
# The AES challenge's own file numbering convention lists six consecutive
# 10-minute preictal segments per continuous 1-hour block preceding a single
# seizure: segment indices 1-6 are block 0, indices 7-12 are block 1, and so
# on. Each file's own 'sequence' field (1-6) records its position within its
# block, which lets this assumption be validated directly against the real
# file contents rather than trusted blindly, the same check used in a
# companion project in this portfolio (seizure_faithfulness_study.py).
# Treating each of the six files as an independent sample, as the original
# version of this script did, overstates the true number of independent
# observations: it means eighteen files could come from as few as three
# actual, independent seizures. This section fixes that by pooling all
# population-state vectors from the six files in a block and computing ONE
# delta-hyperbolicity value per real seizure event, not per file.

def group_preictal_files_by_block(preictal_files: List[str]) -> Dict[int, List[str]]:
    blocks: Dict[int, List[str]] = {}
    for file_path in preictal_files:
        match = re.search(r"segment_(\d+)", os.path.basename(file_path))
        if not match:
            print(f"Warning: could not parse segment number from {file_path}, skipping.")
            continue
        segment_index = int(match.group(1))
        block_id = (segment_index - 1) // 6
        blocks.setdefault(block_id, []).append(file_path)
    return blocks


def validate_block_assumption(preictal_files: List[str], blocks: Dict[int, List[str]]) -> None:
    """Checks each file's own 'sequence' field against the position implied
    by the block grouping above; prints a clear warning rather than failing
    silently if this dataset's numbering convention differs from what public
    solutions to this competition document."""
    checked, mismatches = 0, 0
    for block_id, files in blocks.items():
        for file_path in files:
            loaded = load_mat_segment(file_path)
            if loaded is None or loaded["sequence"] is None:
                continue
            match = re.search(r"segment_(\d+)", os.path.basename(file_path))
            segment_index = int(match.group(1))
            expected_position = ((segment_index - 1) % 6) + 1
            checked += 1
            if expected_position != loaded["sequence"]:
                mismatches += 1
                print(f"  MISMATCH: {os.path.basename(file_path)} sequence={loaded['sequence']}, "
                      f"expected {expected_position} from filename numbering.")

    if checked == 0:
        print("Could not validate block grouping (no readable files with a sequence field).")
    elif mismatches == 0:
        print(f"Block grouping validated on {checked} files: filename numbering matches the "
              f"'sequence' field exactly. {len(blocks)} independent seizure block(s) identified "
              f"from {len(preictal_files)} preictal files.")
    else:
        print(f"WARNING: {mismatches}/{checked} files' sequence field does not match the assumed "
              f"numbering. Fix this grouping before trusting the results below.")


def block_delta_hyperbolicity(file_paths: List[str]) -> Optional[float]:
    """Pools population-state vectors from every file in one seizure block
    (all six 10-minute segments of the same continuous preictal hour) and
    computes ONE delta-hyperbolicity value for the pooled set, representing
    that single real seizure event rather than treating each file as a
    separate observation."""
    all_states = []
    sfreq = None
    for file_path in file_paths:
        loaded = load_mat_segment(file_path)
        if loaded is None:
            continue
        sfreq = loaded["sampling_frequency"]
        states = extract_population_states(loaded["data"], sfreq)
        if states.shape[0] > 0:
            all_states.append(states)

    if not all_states:
        return None

    pooled_states = np.concatenate(all_states, axis=0)
    if pooled_states.shape[0] < 4:
        return None

    dist_matrix = population_state_distance_matrix(pooled_states)
    n_quadruples = min(2000, pooled_states.shape[0] ** 4)
    result = estimate_delta_hyperbolicity(dist_matrix, n_samples=n_quadruples, percentile=95.0)
    return result["delta_estimate"]


# ==============================================================================
# The comparison: does delta-hyperbolicity differ between preictal and
# interictal brain states?
# ==============================================================================

def run_comparison(segment_files: Dict[str, List[str]]) -> Dict:
    # Preictal: aggregated to one value per real seizure block, not per file.
    blocks = group_preictal_files_by_block(segment_files["preictal"])
    print(f"\nGrouping {len(segment_files['preictal'])} preictal files into seizure blocks...")
    validate_block_assumption(segment_files["preictal"], blocks)

    preictal_deltas = []
    for block_id, file_paths in blocks.items():
        delta = block_delta_hyperbolicity(file_paths)
        if delta is not None:
            preictal_deltas.append(delta)

    # Interictal: each file is already a reasonably independent instance,
    # not part of a documented multi-file block structure the way preictal
    # segments are, so per-file treatment remains appropriate here.
    interictal_deltas = []
    for file_path in segment_files["interictal"]:
        loaded = load_mat_segment(file_path)
        if loaded is None:
            continue
        delta = segment_delta_hyperbolicity(loaded["data"], loaded["sampling_frequency"])
        if delta is not None:
            interictal_deltas.append(delta)

    preictal_deltas = np.array(preictal_deltas)
    interictal_deltas = np.array(interictal_deltas)

    from scipy.stats import mannwhitneyu
    if len(preictal_deltas) > 0 and len(interictal_deltas) > 0:
        stat, p_value = mannwhitneyu(preictal_deltas, interictal_deltas, alternative="two-sided")
    else:
        p_value = float("nan")

    return {
        "preictal_deltas": preictal_deltas,
        "interictal_deltas": interictal_deltas,
        "p_value": p_value,
    }


def print_results(results: Dict):
    pre = results["preictal_deltas"]
    inter = results["interictal_deltas"]

    print(f"\n{'':<20}{'N segments':<14}{'Mean delta':<14}{'Std delta':<14}")
    print("-" * 62)
    print(f"{'Preictal':<20}{len(pre):<14}{pre.mean():<14.4f}{pre.std():<14.4f}")
    print(f"{'Interictal':<20}{len(inter):<14}{inter.mean():<14.4f}{inter.std():<14.4f}")
    print(f"\nMann-Whitney U test p-value: {results['p_value']:.4f}")



# ==============================================================================
# Main
# ==============================================================================

def main():
    run_correctness_check()

    segment_files = discover_segment_files(DATA_ROOT, SUBJECT)
    if not segment_files["preictal"] or not segment_files["interictal"]:
        raise RuntimeError(f"No segment files found under {DATA_ROOT}. Check DATA_ROOT and SUBJECT.")

    results = run_comparison(segment_files)
    print_results(results)



if __name__ == "__main__":
    main()


Correctness check: delta-hyperbolicity of a perfect tree metric must be 0.
  Max sampled delta on a perfect tree: 0.00000000  (expected: 0.0)
  PASSED: implementation matches the known analytic property of tree metrics.

Contrast check: delta-hyperbolicity of points on a Euclidean circle (NOT tree-like).
  95th percentile delta on a circle: 0.4252  (expected: > 0, a circle is not tree-like)
  PASSED: non-tree-like structure correctly produces nonzero delta.

Found 18 preictal and 50 interictal files for Patient_1.

Grouping 18 preictal files into seizure blocks...
Block grouping validated on 18 files: filename numbering matches the 'sequence' field exactly. 3 independent seizure block(s) identified from 18 preictal files.

                    N segments    Mean delta    Std delta     
--------------------------------------------------------------
Preictal            3             0.6499        0.3605        
Interictal          50            0.8724        0.0733        

Mann-Whitney U